# Mental Health Dataset — French EDA Pipeline




In [53]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import re
import string
import warnings
from collections import Counter
from itertools import combinations
from pydantic import BaseModel, Field

# ── External libraries ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from wordcloud import WordCloud
import spacy
import emoji
from typing import List, Tuple, Set
from nltk.corpus import stopwords
warnings.filterwarnings("ignore")
print(" All imports OK")
nlp = spacy.load("fr_core_news_sm")  # use the model name as a string

 All imports OK


In [61]:
class PostAnalysis(BaseModel):
    """
    Pydantic model for storing analysis features of a post.
    """
    word_count: int = Field(default=0, description="Number of words")
    char_count: int = Field(default=0, description="Number of characters")
    punct_density: float = Field(default=0.0, description="Punctuation density: total punctuation / word count")
    question_count: int = Field(default=0, description="Number of question marks")
    exclamation_count: int = Field(default=0, description="Number of exclamation marks")
    ellipsis_count: int = Field(default=0, description="Number of ellipsis")
    emoji_count: int = Field(default=0, description="Number of emojis")
    emojis: List[str] = Field(default_factory=list, description="List of emojis")
    emoticon_count: int = Field(default=0, description="Number of emoticons")
    emoticons: List[str] = Field(default_factory=list, description="List of emoticons")
    hashtags: List[str] = Field(default_factory=list, description="List of hashtags")

##  Configuration

In [62]:
class Config:
    """
    Central configuration — change values here, everything else adapts.
    """

    # ── File paths ────────────────────────────────────────────────────────────
    CSV_PATH       = r"C:\\Users\\Admin\\Documents\\FYP\\french dataset\\Dataset\\french_data.csv"          # path to your dataset
    STOPWORDS_FILE = r"C:\Users\Admin\Documents\FYP\french dataset\Dataset\french_stpwords.txt"     # path to French stopwords file
    OUTPUT_DIR     = r"MyResults"                   # folder for saved plots

    # ── Column names ─────────────────────────────────────────────────────────
    LANGUAGE_COL   = "language"
    LANGUAGE_VALUE = "French"
    TEXT_COL       = "text"
    LABEL_COL      = "mental_state"

    # ── Plot settings ─────────────────────────────────────────────────────────
    BG      = "#F9F9F9"    # background color for all plots
    DPI     = 150          # resolution when saving figures
    PALETTE = "Set2"       # seaborn color palette

    # ── Analysis settings ─────────────────────────────────────────────────────
    TOP_N_WORDS = 20       # top N most frequent words per label
    TOP_N_COOC  = 20       # top N word co-occurrence pairs per label

cfg = Config()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
print(f"Config ready — output folder: '{cfg.OUTPUT_DIR}'")


Config ready — output folder: 'MyResults'


##  Plot Helper

In [63]:
class PlotHelper:
    """
    Centralizes styling and saving of all matplotlib figures.
    """

    def __init__(self, cfg: Config):
        self.cfg = cfg
        plt.rcParams.update({
            "figure.facecolor" : cfg.BG,
            "axes.facecolor"   : cfg.BG,
            "axes.spines.top"  : False,
            "axes.spines.right": False,
            "font.size"        : 11,
        })

    def save(self, filename: str) -> str:
        """Save the current figure and return its path."""
        path = os.path.join(self.cfg.OUTPUT_DIR, filename)
        plt.tight_layout()
        plt.savefig(path, dpi=self.cfg.DPI, bbox_inches="tight")
        plt.close()
        print(f"  [SAVED] {filename}")
        return path

    @staticmethod
    def safe_name(text: str) -> str:
        """Convert a string to a safe filename (removes forbidden chars)."""
        return re.sub(r'[\\/*?"<>|]+', "_", str(text)).strip()

helper = PlotHelper(cfg)
print(" PlotHelper ready")


 PlotHelper ready


##  Data Loader

In [64]:
class DataLoader:
    """Loads the CSV and filters to French rows only."""

    def __init__(self, cfg: Config):
        self.cfg = cfg

    def load(self) -> pd.DataFrame:
        df = pd.read_csv(self.cfg.CSV_PATH)
        print(f"[DataLoader] Total rows loaded       : {len(df)}")

        # Case-insensitive filter on language column
        mask = (
            df[self.cfg.LANGUAGE_COL].str.strip().str.lower()
            == self.cfg.LANGUAGE_VALUE.lower()
        )
        df = df[mask].copy().reset_index(drop=True)
        print(f"[DataLoader] French rows kept         : {len(df)}")
        return df


##  Text Cleaner

In [65]:
nlp = spacy.load("fr_core_news_sm", disable=["ner", "parser"])

# Merge spaCy + NLTK + custom stopwords
SPACY_STOPS = nlp.Defaults.stop_words.copy()
NLTK_STOPS = set(stopwords.words("french"))
CUSTOM_STOPS = {"bonjour", "salut", "svp", "merci", "plait"}
STOPWORDS = SPACY_STOPS | NLTK_STOPS | CUSTOM_STOPS

class TextCleaner:
    def __init__(self) -> None:
        self.nlp = nlp
        self.stopwords_set: Set[str] = STOPWORDS

        self.emoji_regex = (
            r'[\U0001F600-\U0001F64F]|'
            r'[\U0001F300-\U0001F5FF]|'
            r'[\U0001F680-\U0001F6FF]|'
            r'[\U00002600-\U000026FF]|'
            r'[\U00002700-\U000027BF]|'
            r'[\U0001F900-\U0001F9FF]|'
            r'[\U0001FA00-\U0001FA6F]|'
            r'[\U0001FA70-\U0001FAFF]'
        )

    def remove_emojis(self, text: str) -> str:
        text_no_emoji = emoji.replace_emoji(text, replace="")
        return re.sub(self.emoji_regex, "", text_no_emoji)

    def replace_urls(self, text: str) -> str:
        return re.sub(r'https?://\S+|www\.\S+', ' URL ', text)

    def replace_mentions(self, text: str) -> str:
        return re.sub(r'@\w+', ' PEOPLE ', text)

    def extract_hashtags(self, text: str) -> Tuple[List[str], str]:
        hashtags = re.findall(r'#\w+', text)
        text_without_hashtags = re.sub(r'#\w+', '', text)
        return hashtags, text_without_hashtags

    def standardize_text(self, text: str) -> str:
        return text.lower().replace('\n', ' ').replace('\r', ' ')

    def tokenize(self, cleaned_text: str) -> List[str]:
        doc = self.nlp(cleaned_text)
        return [token.text.lower() for token in doc if token.is_alpha]

    def lemmatize(self, tokens: List[str]) -> List[str]:
        doc = self.nlp(" ".join(tokens))
        return [token.lemma_.lower() for token in doc if token.is_alpha]

    def remove_stopwords(self, tokens: List[str]) -> List[str]:
        return [t for t in tokens if t not in self.stopwords_set]

    def clean_text(self, text: str) -> Tuple[str, List[str]]:
        text = self.standardize_text(text)
        text = self.remove_emojis(text)
        text = self.replace_urls(text)
        text = self.replace_mentions(text)
        hashtags, text = self.extract_hashtags(text)
        text = re.sub(r"[^\w\s!?\-']", "", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text, hashtags

    def preprocess(self, cleaned_text: str) -> List[str]:
        tokens = self.tokenize(cleaned_text)
        lemmas = self.lemmatize(tokens)
        return self.remove_stopwords(lemmas)


The FeatureExtractor can help to analyze things like:

emoji usage by mental state
punctuation patterns
emotional intensity in posts

In [68]:
class FeatureExtractor:
    """Extracts features from text data."""

    def __init__(self, processor: TextProcessor) -> None:
        self.processor = processor

    def count_punctuation(self, text: str) -> Dict[str, int]:
        return {
            'question_count': text.count('?'),
            'exclamation_count': text.count('!'),
            'ellipsis_count': text.count('...')
        }

    def detect_emojis_combined(self, text: str) -> Tuple[List[str], int]:
        found_emojis = [item['emoji'] for item in emoji.emoji_list(text)]
        standardized = [emoji.demojize(e) for e in found_emojis]
        regex_matches = re.findall(self.processor.emoji_regex, text)
        for match in regex_matches:
            name = emoji.demojize(match)
            if name not in standardized:
                standardized.append(name)
        return standardized, len(standardized)

    def detect_emoticons(self, text: str) -> Tuple[List[str], int]:
        emoticons: List[str] = []
        for pattern in self.processor.emoticon_patterns:
            matches = re.findall(pattern, text, re.IGNORECASE)
            emoticons.extend(matches)
        return emoticons, len(emoticons)

    def extract_ngrams(self, tokens: List[str], n: int) -> List[str]:
        if len(tokens) < n:
            return []
        ngram_list = [" ".join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]
        return [ng for ng in ngram_list if all(word.isalpha() for word in ng.split())]

    def compute_post_features_from_doc(self, original_text: str, cleaned_text: str, doc) -> PostAnalysis:
        word_count = sum(1 for token in doc if token.is_alpha)
        char_count = len(cleaned_text)
        punct_counts = self.count_punctuation(cleaned_text)
        total_punct = sum(punct_counts.values())
        punct_density = total_punct / word_count if word_count > 0 else 0.0
        emojis, emoji_count = self.detect_emojis_combined(original_text)
        emoticons, emoticon_count = self.detect_emoticons(original_text)
        return PostAnalysis(
            word_count=word_count, char_count=char_count, punct_density=punct_density,
            question_count=punct_counts.get('question_count', 0),
            exclamation_count=punct_counts.get('exclamation_count', 0),
            ellipsis_count=punct_counts.get('ellipsis_count', 0),
            emoji_count=emoji_count, emojis=emojis,
            emoticon_count=emoticon_count, emoticons=emoticons, hashtags=[]
        )

##  Load & Clean the Data

In [69]:
"""# Load 
df_raw = DataLoader(cfg).load()

# Clean 
cleaner = TextCleaner()
df       = cleaner.fit_transform(df_raw, cfg.TEXT_COL)

# Save cleaned CSV 
clean_path = os.path.join(cfg.OUTPUT_DIR, "C:\\Users\\Admin\\Documents\\FYP\\french dataset\\Dataset\\french_data.csv")
df.to_csv(clean_path, index=False, encoding="utf-8-sig")
print(f"\n Cleaned CSV saved → {clean_path}")
df.head(3)
"""

'# Load \ndf_raw = DataLoader(cfg).load()\n\n# Clean \ncleaner = TextCleaner()\ndf       = cleaner.fit_transform(df_raw, cfg.TEXT_COL)\n\n# Save cleaned CSV \nclean_path = os.path.join(cfg.OUTPUT_DIR, "C:\\Users\\Admin\\Documents\\FYP\\french dataset\\Dataset\\french_data.csv")\ndf.to_csv(clean_path, index=False, encoding="utf-8-sig")\nprint(f"\n Cleaned CSV saved → {clean_path}")\ndf.head(3)\n'

##   Dataset Size

In [2]:
"""class DatasetSize:
"""    """Req 0 — General statistics about the dataset (no plot needed)."""
"""
    def __init__(self, cfg: Config):
        self.cfg = cfg

    def analyse(self, df: pd.DataFrame) -> dict:
        stats = {
            "Total rows (French)"  : len(df),
            "Unique texts"         : df["text_clean"].nunique(),
            "Columns"              : len(df.columns),
            "Labels"               : df[self.cfg.LABEL_COL].nunique(),
            "Avg word count"       : round(df["word_count"].mean(), 2),
            "Max word count"       : df["word_count"].max(),
            "Min word count"       : df["word_count"].min(),
            "Avg char count"       : round(df["char_count"].mean(), 2),
            "Missing values total" : int(df.isnull().sum().sum()),
        }
        print("\n[DatasetSize]")
        for k, v in stats.items():
            print(f"  {k:<28}: {v}")
        return stats

stats = DatasetSize(cfg).analyse(df)
"""

'\n    def __init__(self, cfg: Config):\n        self.cfg = cfg\n\n    def analyse(self, df: pd.DataFrame) -> dict:\n        stats = {\n            "Total rows (French)"  : len(df),\n            "Unique texts"         : df["text_clean"].nunique(),\n            "Columns"              : len(df.columns),\n            "Labels"               : df[self.cfg.LABEL_COL].nunique(),\n            "Avg word count"       : round(df["word_count"].mean(), 2),\n            "Max word count"       : df["word_count"].max(),\n            "Min word count"       : df["word_count"].min(),\n            "Avg char count"       : round(df["char_count"].mean(), 2),\n            "Missing values total" : int(df.isnull().sum().sum()),\n        }\n        print("\n[DatasetSize]")\n        for k, v in stats.items():\n            print(f"  {k:<28}: {v}")\n        return stats\n\nstats = DatasetSize(cfg).analyse(df)\n'

In [ ]:

class TextAnalyzer:
    """
    Analyzes processed text and computes linguistic features.
    """

    def __init__(self, processor: TextProcessor, feature_extractor: FeatureExtractor) -> None:
        """
        Initialize the TextAnalyzer with processor and feature extractor.

        Args:
            processor: The TextProcessor instance.
            feature_extractor: The FeatureExtractor instance.
        """
        self.processor = processor
        self.feature_extractor = feature_extractor
        self.labels: List[str] = []
        self.label_tokens: Dict[str, List[List[str]]] = {}

    def compute_vocabulary_diversity(self, tokens: List[str]) -> float:
        """
        Compute the type-token ratio (vocabulary diversity) for a list of tokens.

        Args:
            tokens: List of token strings.

        Returns:
            The TTR as a float.
        """
        if len(tokens) == 0:
            return 0.0
        # sets allow for only unique tokens
        # so we can count how many unique tokens there are
        return len(set(tokens)) / len(tokens)

    def compute_normalized_punctuation(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Compute normalized punctuation counts per sentence for each label.

        Args:
            df: The dataframe with text and punctuation counts.

        Returns:
            A dataframe with normalized punctuation statistics.
        """
        punct_cols = ['question_count', 'exclamation_count', 'ellipsis_count']
        sentence_counts = df['text'].apply(
            lambda text: max(len(sent_tokenize(text)), 1)
        ) # split text into sentences 
        # avoid division by zero, treat empty text as 1 sentence
        summary_data = []
        # for each label we compute the average punctuation count per sentence 
        # by dividing the average punctuation count by the average sentence count for that label
        for label in df['status'].unique():
            # create a dataframe for the current label
            label_data = df[df['status'] == label]
            avg_sent_len = sentence_counts[label_data.index].mean()
            # Normalization
            normalized_punct = {}
            for col in punct_cols:
                avg_count = label_data[col].mean()
                normalized_value = avg_count / avg_sent_len if avg_sent_len > 0 else 0
                # exclamation_count -> Exclamation
                normalized_punct[col.replace('_count', '').title()] = normalized_value
            normalized_punct['Label'] = label
            # save result into the list
            summary_data.append(normalized_punct)
            
        # converts summary_data to a DataFrame where Label is the row name
        return pd.DataFrame(summary_data).set_index('Label')

    def compute_label_top_ngrams(
        self,
        df: pd.DataFrame,
        text_column: str,
        label_column: str,
        n: int,
        top_k: int = 10
    ) -> Dict[str, List[Tuple[str, int]]]:
        """
        Compute the most frequent n-grams for each label.
        1. get tokens
        2. compute n-grams
        3. count frequency
        Uses cached tokens if available, otherwise reprocesses.
        """
        # Example output: {'Anxiety': [('feel sad', 15), ('not happy', 10)], 
        # 'Depression': [('feel hopeless', 20), ...], ...}
        label_ngrams: Dict[str, List[Tuple[str, int]]] = {}

        for label in df[label_column].unique():
            label_mask = df[label_column] == label
            # Counter is a dict subclass for counting hashable objects.
            # It counts the frequency of each n-gram.
            ngram_counter: Counter[str] = Counter()

            # Use cached tokens if available
            if 'tokens' in df.columns:
                # in this case, we already have the list of words
                for tokens in df[label_mask]['tokens']:
                    # some rows might have missing or malformed tokens, 
                    # so we check if it's a list before processing
                    if isinstance(tokens, list):
                        ngram_counter.update(
                            self.feature_extractor.extract_ngrams(tokens, n)
                        )
            else:
                # Fallback: reprocess (slower)
                # We don't have the list.
                # We must call the preprocess() function now.
                for text in df[label_mask][text_column]:
                    tokens = self.processor.preprocess(text)
                    ngram_counter.update(
                        self.feature_extractor.extract_ngrams(tokens, n)
                    )

            label_ngrams[label] = ngram_counter.most_common(top_k)

        return label_ngrams

    def compute_shared_vocabulary(self, df: pd.DataFrame) -> Dict[str, Set[str]]:
        """
            Compute vocabulary (unique words) for each label.
            Uses cached tokens if available, otherwise reprocesses.
        """
        label_vocab: Dict[str, Set[str]] = {}

        for label in df['status'].unique():
            label_mask = df['status'] == label
            vocab: Set[str] = set()

            # Use cached tokens if available
            if 'tokens' in df.columns:
                for tokens in df[label_mask]['tokens']:
                    if isinstance(tokens, list):
                        vocab.update(t for t in tokens if t and t.isalpha())
            else:
                # Fallback: reprocess (slower)
                text_col = 'cleaned_text' if 'cleaned_text' in df.columns else 'text'
                for text in df[label_mask][text_col]:
                    if not isinstance(text, str):
                        continue
                    tokens = self.processor.preprocess(text)
                    vocab.update(t for t in tokens if t and t.isalpha())

            label_vocab[label] = vocab

        return label_vocab


##   Label Distribution

In [5]:
class LabelDistribution:
    """Req 1 — Bar chart + pie chart of label counts."""

    def __init__(self, cfg: Config, helper: PlotHelper):
        self.cfg    = cfg
        self.helper = helper

    def analyse(self, df: pd.DataFrame) -> str:
        counts = df[self.cfg.LABEL_COL].value_counts()

        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        fig.suptitle("Label Distribution — mental_state", fontsize=14, fontweight="bold")

        # Bar chart
        sns.barplot(x=counts.values, y=counts.index.astype(str),
                    palette=self.cfg.PALETTE, ax=axes[0])
        axes[0].set_xlabel("Count")
        axes[0].set_title("Count per Label")
        for bar, val in zip(axes[0].patches, counts.values):
            axes[0].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                         str(val), va="center", fontsize=9)

        # Pie chart
        axes[1].pie(counts.values, labels=counts.index,
                    autopct="%1.1f%%",
                    colors=sns.color_palette(self.cfg.PALETTE, len(counts)),
                    startangle=140)
        axes[1].set_title("Proportion per Label")

        return self.helper.save("01_label_distribution.png")

LabelDistribution(cfg, helper).analyse(df)


NameError: name 'cfg' is not defined

##  Text Length Analysis

In [ ]:
class TextLengthAnalysis:
    """Req 2 — Histogram of word count + boxplot of char count by label."""

    def __init__(self, cfg: Config, helper: PlotHelper):
        self.cfg    = cfg
        self.helper = helper

    def analyse(self, df: pd.DataFrame) -> str:
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        fig.suptitle("Text Length Analysis", fontsize=14, fontweight="bold")

        # Histogram + KDE of word count
        axes[0].hist(df["word_count"], bins=30, color="#4C9BE8",
                     edgecolor="white", density=True, alpha=0.7, label="Word Count")
        df["word_count"].plot.kde(ax=axes[0], color="darkblue", linewidth=2, label="KDE")
        axes[0].axvline(df["word_count"].mean(), color="red", linestyle="--",
                        label=f"Mean={df['word_count'].mean():.1f}")
        axes[0].set_title("Word Count Distribution")
        axes[0].set_xlabel("Words per text")
        axes[0].set_ylabel("Density")
        axes[0].legend()

        # Boxplot of char count by label
        order = (df.groupby(self.cfg.LABEL_COL)["char_count"]
                   .median().sort_values(ascending=False).index)
        sns.boxplot(data=df, x=self.cfg.LABEL_COL, y="char_count",
                    order=order, palette=self.cfg.PALETTE, ax=axes[1])
        axes[1].set_title("Char Count by Label")
        axes[1].set_xlabel("")
        plt.setp(axes[1].get_xticklabels(), rotation=30, ha="right")

        return self.helper.save("02_text_length.png")

TextLengthAnalysis(cfg, helper).analyse(df)


  [SAVED] 02_text_length.png


'MyResults\\02_text_length.png'

##  Punctuation Usage

In [ ]:
class PunctuationAnalysis:
    """Req 3 — Histogram of punct per text + avg punct per label bar chart."""

    def __init__(self, cfg: Config, helper: PlotHelper):
        self.cfg    = cfg
        self.helper = helper

    def analyse(self, df: pd.DataFrame) -> str:
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        fig.suptitle("Punctuation Usage", fontsize=14, fontweight="bold")

        # Histogram
        axes[0].hist(df["punct_count"], bins=25, color="#E87B4C", edgecolor="white")
        axes[0].axvline(df["punct_count"].mean(), color="red", linestyle="--",
                        label=f"Mean={df['punct_count'].mean():.1f}")
        axes[0].set_title("Punct Count Distribution")
        axes[0].set_xlabel("Punctuation marks per text")
        axes[0].legend()

        # Avg punctuation per label
        avg = (df.groupby(self.cfg.LABEL_COL)["punct_count"]
                  .mean().sort_values(ascending=False))
        sns.barplot(x=avg.values, y=avg.index.astype(str),
                    palette=self.cfg.PALETTE, ax=axes[1])
        axes[1].set_title("Avg Punct Count by Label")
        axes[1].set_xlabel("Avg count")

        return self.helper.save("03_punctuation.png")

PunctuationAnalysis(cfg, helper).analyse(df)


NameError: name 'helper' is not defined

##   Word Clouds per Label

In [ ]:
class WordCloudAnalysis:
    """Req 4 — One word cloud per label (using text_nostop)."""

    def __init__(self, cfg: Config, helper: PlotHelper):
        self.cfg    = cfg
        self.helper = helper

    def analyse(self, df: pd.DataFrame) -> list:
        labels = df[self.cfg.LABEL_COL].unique()
        n      = len(labels)
        cols   = min(3, n)
        rows   = (n + cols - 1) // cols

        fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 4))
        axes = np.array(axes).flatten()
        fig.suptitle("Word Clouds per Label", fontsize=15, fontweight="bold")

        cmaps = ["Blues", "Reds", "Greens", "Purples", "Oranges", "YlOrBr"]

        for i, label in enumerate(labels):
            text = " ".join(df[df[self.cfg.LABEL_COL] == label]["text_nostop"])
            if not text.strip():
                axes[i].axis("off")
                continue

            wc = WordCloud(
                width=600, height=350,
                background_color="white",
                colormap=cmaps[i % len(cmaps)],
                max_words=100,
                collocations=False,
            ).generate(text)

            axes[i].imshow(wc, interpolation="bilinear")
            axes[i].axis("off")
            axes[i].set_title(str(label), fontsize=12, fontweight="bold")

        for j in range(i + 1, len(axes)):
            axes[j].axis("off")

        path = self.helper.save("04_wordclouds_per_label.png")
        return [path]

WordCloudAnalysis(cfg, helper).analyse(df)


  [SAVED] 04_wordclouds_per_label.png


['MyResults\\04_wordclouds_per_label.png']

##  Word Co-occurrence per Label

In [ ]:
class CoOccurrenceAnalysis:
    """Req 5 — Top word pairs that appear together per label (horizontal bar chart)."""

    def __init__(self, cfg: Config, helper: PlotHelper):
        self.cfg    = cfg
        self.helper = helper

    def _cooccurrence(self, texts, top_n):
        co = Counter()
        for sentence in texts:
            words = list(set(sentence.split()))
            for pair in combinations(sorted(words), 2):
                co[pair] += 1
        return co.most_common(top_n)

    def analyse(self, df: pd.DataFrame) -> list:
        paths  = []
        labels = df[self.cfg.LABEL_COL].unique()

        for label in labels:
            texts = df[df[self.cfg.LABEL_COL] == label]["text_nostop"]
            pairs = self._cooccurrence(texts, self.cfg.TOP_N_COOC)
            if not pairs:
                continue

            pair_labels = [f"{a} & {b}" for (a, b), _ in pairs]
            counts      = [c for _, c in pairs]

            fig, ax = plt.subplots(figsize=(10, 6))
            sns.barplot(x=counts, y=pair_labels, palette="mako", ax=ax)
            ax.set_title(f"Top Word Co-occurrences — {label}", fontsize=13, fontweight="bold")
            ax.set_xlabel("Co-occurrence count")

            fname = f"05_cooccurrence_{self.helper.safe_name(label)}.png"
            paths.append(self.helper.save(fname))

        return paths

CoOccurrenceAnalysis(cfg, helper).analyse(df)


  [SAVED] 05_cooccurrence_Healthy.png
  [SAVED] 05_cooccurrence_Unhealthy.png


['MyResults\\05_cooccurrence_Healthy.png',
 'MyResults\\05_cooccurrence_Unhealthy.png']

##   Common Words per Label

In [ ]:
from collections import Counter
from itertools import islice

def get_ngrams(words: list, n: int) -> list:
    """Generate n-grams from a list of words."""
    return [" ".join(words[i:i+n]) for i in range(len(words) - n + 1)]


class CommonWordsAnalysis:
    """Req 6 — Top N most frequent words, bigrams, and trigrams per label."""

    def __init__(self, cfg: Config, helper: PlotHelper):
        self.cfg    = cfg
        self.helper = helper

    def _plot_ngrams(
        self,
        df: pd.DataFrame,
        n: int,
        title_prefix: str,
        filename: str,
    ) -> str:
        labels = df[self.cfg.LABEL_COL].unique()
        cols   = min(2, len(labels))
        rows   = (len(labels) + cols - 1) // cols

        fig, axes = plt.subplots(rows, cols, figsize=(cols * 8, rows * 5))
        axes = np.array(axes).flatten()
        fig.suptitle(
            f"Top {self.cfg.TOP_N_WORDS} {title_prefix} per Label",
            fontsize=14, fontweight="bold",
        )

        for i, label in enumerate(labels):
            words  = " ".join(df[df[self.cfg.LABEL_COL] == label]["text_nostop"]).split()
            ngrams = get_ngrams(words, n)
            freq   = Counter(ngrams).most_common(self.cfg.TOP_N_WORDS)

            if not freq:
                axes[i].axis("off")
                continue

            w, c = zip(*freq)
            sns.barplot(x=list(c), y=list(w), palette="rocket", ax=axes[i])
            axes[i].set_title(str(label), fontsize=11, fontweight="bold")
            axes[i].set_xlabel("Frequency")
            axes[i].set_ylabel("")

        for j in range(i + 1, len(axes)):
            axes[j].axis("off")

        plt.tight_layout()
        return self.helper.save(filename)

    def analyse(self, df: pd.DataFrame) -> list:
        paths = []

        # Unigrams (original behaviour)
        paths.append(self._plot_ngrams(
            df, n=1,
            title_prefix="Common Words",
            filename="06_common_words_per_label.png",
        ))

        # Bigrams
        paths.append(self._plot_ngrams(
            df, n=2,
            title_prefix="Bigrams",
            filename="06_bigrams_per_label.png",
        ))

        # Trigrams
        paths.append(self._plot_ngrams(
            df, n=3,
            title_prefix="Trigrams",
            filename="06_trigrams_per_label.png",
        ))

        return paths


CommonWordsAnalysis(cfg, helper).analyse(df)

  [SAVED] 06_common_words_per_label.png
  [SAVED] 06_bigrams_per_label.png
  [SAVED] 06_trigrams_per_label.png


['MyResults\\06_common_words_per_label.png',
 'MyResults\\06_bigrams_per_label.png',
 'MyResults\\06_trigrams_per_label.png']

##  Emoji & Emoticon Analysis

In [ ]:
class EmojiEmoticonAnalysis:
    """
    Req 7 & 8 — Emoji and emoticon usage analysis.
    Produces:
      - Bar chart: avg emoji count per label
      - Bar chart: avg emoticon count per label
      - Histogram: distribution of emoji counts
      - Histogram: distribution of emoticon counts
    """

    def __init__(self, cfg: Config, helper: PlotHelper):
        self.cfg    = cfg
        self.helper = helper

    def analyse(self, df: pd.DataFrame) -> str:
        fig, axes = plt.subplots(2, 2, figsize=(13, 10))
        fig.suptitle("Emoji & Emoticon Analysis", fontsize=14, fontweight="bold")

        # ── Top-left: emoji count distribution (histogram) ────────────────────
        axes[0, 0].hist(df["emoji_count"], bins=20, color="#F4A460", edgecolor="white")
        axes[0, 0].axvline(df["emoji_count"].mean(), color="red", linestyle="--",
                           label=f"Mean={df['emoji_count'].mean():.2f}")
        axes[0, 0].set_title("Emoji Count Distribution")
        axes[0, 0].set_xlabel("Emojis per text")
        axes[0, 0].legend()

        # ── Top-right: emoticon count distribution (histogram) ────────────────
        axes[0, 1].hist(df["emoticon_count"], bins=20, color="#87CEEB", edgecolor="white")
        axes[0, 1].axvline(df["emoticon_count"].mean(), color="red", linestyle="--",
                           label=f"Mean={df['emoticon_count'].mean():.2f}")
        axes[0, 1].set_title("Emoticon Count Distribution")
        axes[0, 1].set_xlabel("Emoticons per text")
        axes[0, 1].legend()

        # ── Bottom-left: avg emoji count per label (bar chart) ────────────────
        avg_emoji = (df.groupby(self.cfg.LABEL_COL)["emoji_count"]
                       .mean().sort_values(ascending=False))
        sns.barplot(x=avg_emoji.values, y=avg_emoji.index.astype(str),
                    palette=self.cfg.PALETTE, ax=axes[1, 0])
        axes[1, 0].set_title("Avg Emoji Count by Label")
        axes[1, 0].set_xlabel("Avg emoji count")

        # ── Bottom-right: avg emoticon count per label (bar chart) ────────────
        avg_emot = (df.groupby(self.cfg.LABEL_COL)["emoticon_count"]
                      .mean().sort_values(ascending=False))
        sns.barplot(x=avg_emot.values, y=avg_emot.index.astype(str),
                    palette=self.cfg.PALETTE, ax=axes[1, 1])
        axes[1, 1].set_title("Avg Emoticon Count by Label")
        axes[1, 1].set_xlabel("Avg emoticon count")

        return self.helper.save("07_08_emoji_emoticon.png")

EmojiEmoticonAnalysis(cfg, helper).analyse(df)


  [SAVED] 07_08_emoji_emoticon.png


'MyResults\\07_08_emoji_emoticon.png'

##  Pipeline Summary

In [ ]:
print("=" * 55)
print("  FRENCH EDA PIPELINE — COMPLETE")
print("=" * 55)

saved = [f for f in os.listdir(cfg.OUTPUT_DIR) if f.endswith(".png")]
print(f"\n📊 {len(saved)} plots saved to '{cfg.OUTPUT_DIR}/':")
for f in sorted(saved):
    print(f"   • {f}")

print(f"\n📄 Cleaned CSV : {cfg.OUTPUT_DIR}/french_cleaned.csv")
print("\n✅ All done!")


  FRENCH EDA PIPELINE — COMPLETE

📊 10 plots saved to 'MyResults/':
   • 01_label_distribution.png
   • 02_text_length.png
   • 03_punctuation.png
   • 04_wordclouds_per_label.png
   • 05_cooccurrence_Healthy.png
   • 05_cooccurrence_Unhealthy.png
   • 06_bigrams_per_label.png
   • 06_common_words_per_label.png
   • 06_trigrams_per_label.png
   • 07_08_emoji_emoticon.png

📄 Cleaned CSV : MyResults/french_cleaned.csv

✅ All done!
